In [1]:
!pip install -q ultralytics 
!pip install torch
!pip install easyocr
!pip install gitpython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 29.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 25.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 15.6 MB/s eta 0:00:00


In [ ]:
# Mount Drive
import json
import os
import sys
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
import cv2
from git import List
import easyocr
reader = easyocr.Reader(['en'])

from ultralytics import YOLO
from google.colab import drive
drive.mount('/content/drive')

# Clone repo if needed
# if not os.path.exists("/content/Magma"):
#     !git clone https://github.com/microsoft/Magma.git /content/Magma

# Install deps (light version)


Mounted at /content/drive


In [9]:
BASE_DIR = "/content/drive/MyDrive/Dataset/Testing"
TESTCASE_DIR = f"{BASE_DIR}/testcase"
OUTPUT_DIR = f"{BASE_DIR}/annotated"
TEST_CASES_JSON = f"{BASE_DIR}/test_cases.json"
OMNIPARSER_MODEL_PATH = f"{BASE_DIR}/weights/icon_detect/model.pt"

MIN_BOX_AREA = 0.0
OVERLAP_IOU_THRESHOLD = 0.7

# OmniParser params
BOX_THRESHOLD = 0.25
MAX_MARKS = 100
MARK_FONT_MIN = 12
MARK_FONT_MAX = 14
# =======================================

In [4]:
font_path = os.path.abspath(os.path.join(BASE_DIR, "arial.ttf"))


class MarkHelper:
    def __init__(self):    
        self.markSize_dict = {}
        self.font_dict = {}
        self.min_font_size = 20 # 1 in v1
        self.max_font_size = 30
        self.max_font_proportion = 0.04 # 0.032 in v1

    def __get_markSize(self, text, image_height, image_width, font):
        im = Image.new('RGB', (image_width, image_height))
        draw = ImageDraw.Draw(im)
        _, _, width, height = draw.textbbox((0, 0), text=text, font=font)
        return height, width

    def _setup_new_font(self, image_height, image_width):
        key = f"{image_height}_{image_width}"
        # print(f"Setting up new font for image size: {key}")
        
        # setup the font
        fontsize = self.min_font_size
        font = ImageFont.truetype(font_path, fontsize)
        # font = ImageFont.load_default(size=fontsize)
        while min(self.__get_markSize("555", image_height, image_width, font)) < min(self.max_font_size, self.max_font_proportion * min(image_height, image_width)):
            # iterate until the text size is just larger than the criteria
            fontsize += 1
            font = ImageFont.truetype(font_path, fontsize)
            # font = ImageFont.load_default(size=fontsize)
        self.font_dict[key] = font

        # setup the markSize dict
        markSize_3digits = self.__get_markSize('555', image_height, image_width, font)
        markSize_2digits = self.__get_markSize('55', image_height, image_width, font)
        markSize_1digit = self.__get_markSize('5', image_height, image_width, font)
        self.markSize_dict[key] = {
            1: markSize_1digit,
            2: markSize_2digits,
            3: markSize_3digits
        }

    def get_font(self, image_height, image_width):
        key = f"{image_height}_{image_width}"
        if key not in self.font_dict:
            self._setup_new_font(image_height, image_width)
        return self.font_dict[key]
        
    def get_mark_size(self, text_str, image_height, image_width):
        """Get the font size for the given image dimensions."""
        key = f"{image_height}_{image_width}"
        if key not in self.markSize_dict:
            self._setup_new_font(image_height, image_width)

        largest_size = self.markSize_dict[key].get(3, None)
        text_h, text_w = self.markSize_dict[key].get(len(text_str), largest_size) # default to the largest size if the text is too long
        return text_h, text_w
    

def __calculate_iou(box1, box2, return_area=False):
    """
    Calculate the Intersection over Union (IoU) of two bounding boxes.
    :param box1: Tuple of (y, x, h, w) for the first bounding box
    :param box2: Tuple of (y, x, h, w) for the second bounding box
    :return: IoU value
    """
    y1, x1, h1, w1 = box1
    y2, x2, h2, w2 = box2

    # Calculate the intersection area
    y_min = max(y1, y2)
    x_min = max(x1, x2)
    y_max = min(y1 + h1, y2 + h2)
    x_max = min(x1 + w1, x2 + w2)

    intersection_area = max(0, y_max - y_min) * max(0, x_max - x_min)

    # Compute the area of both bounding boxes
    box1_area = h1 * w1
    box2_area = h2 * w2

    # Calculate the IoU
    # iou = intersection_area / box1_area + box2_area - intersection_area
    iou = intersection_area / (min(box1_area, box2_area) + 0.0001)

    if return_area:
        return iou, intersection_area
    return iou

def __calculate_nearest_corner_distance(box1, box2):
    """Calculate the distance between the nearest edge or corner of two bounding boxes."""
    y1, x1, h1, w1 = box1
    y2, x2, h2, w2 = box2
    corners1 = np.array([
        [y1, x1],
        [y1, x1 + w1],
        [y1 + h1, x1],
        [y1 + h1, x1 + w1]
    ])
    corners2 = np.array([
        [y2, x2],
        [y2, x2 + w2],
        [y2 + h2, x2],
        [y2 + h2, x2 + w2]
    ])
    # Calculate pairwise distances between corners
    distances = np.linalg.norm(corners1[:, np.newaxis] - corners2, axis=2)

    # Find the minimum distance
    min_distance = np.min(distances)
    return min_distance





def _find_least_overlapping_corner(bbox, bboxes, drawn_boxes, text_size, image_size):
    """Find the corner with the least overlap with other bboxes.
    Args:
        bbox: (y, x, h, w) The bounding box to place the text on.
        bboxes: [(y, x, h, w)] The list of bounding boxes to compare against.
        drawn_boxes: [(y, x, h, w)] The list of bounding boxes that have already been drawn on.
        text_size: (height, width) The size of the text to be drawn.
        image_size: (height, width) The size of the image.
    """
    y, x, h, w = bbox
    h_text, w_text = text_size
    image_height, image_width = image_size
    corners = [
        # top-left
        (y - h_text, x),
        # top-right
        (y - h_text, x + w - w_text),
        # right-top
        (y, x + w),
        # right-bottom
        (y + h - h_text, x + w),
        # bottom-right
        (y + h, x + w - w_text),
        # bottom-left
        (y + h, x),
        # left-bottom
        (y + h - h_text, x - w_text),
        # left-top
        (y, x - w_text),
        ]
    best_corner = corners[0]
    max_flag = float('inf')

    for corner in corners:
        corner_bbox = (corner[0], corner[1], h_text, w_text)
        # if the corner is out of the image, skip
        if corner[0] < 0 or corner[1] < 0 or corner[0] + h_text > image_height or corner[1] + w_text > image_width:
            continue
        max_iou = - (image_width + image_height)
        # 找到关于这个角最差的 case
        # given the current corner, find the larget iou with other bboxes.
        for other_bbox in bboxes + drawn_boxes:
            if np.array_equal(bbox, other_bbox):
                continue
            iou = __calculate_iou(corner_bbox, other_bbox, return_area=True)[1]
            max_iou = max(max_iou, iou - 0.0001 * __calculate_nearest_corner_distance(corner_bbox, other_bbox))
        # the smaller the max_IOU, the better the corner
        # 取最差的值 相对最好的那个角
        if max_iou < max_flag:
            max_flag = max_iou
            best_corner = corner

    return best_corner

def plot_boxes_with_marks(
    image: Image.Image,
    bboxes, # (y, x, h, w)
    mark_helper: MarkHelper,
    linewidth=2,
    alpha=0,
    edgecolor=None,
    fn_save=None,
    normalized_to_pixel=True,
    add_mark=True
) -> np.ndarray:
    """Plots bounding boxes on an image with marks attached to the edges of the boxes where no overlap with other boxes occurs.
    Args:
        image: The image to plot the bounding boxes on.
        bboxes: A 2D int array of shape (num_boxes, 4), where each row represents a bounding box: (y_top_left, x_top_left, box_height, box_width). If normalized_to_pixel is True, the values are float and are normalized with the image size. If normalized_to_pixel is False, the values are int and are in pixel.
    """
    # Then modify the drawing code
    draw = ImageDraw.Draw(image)

    # draw boxes on the image
    image_width, image_height = image.size

    if normalized_to_pixel:
        bboxes = [(int(y * image_height), int(x * image_width), int(h * image_height), int(w * image_width)) for y, x, h, w in bboxes]

    for box in bboxes:
        y, x, h, w = box
        draw.rectangle([x, y, x + w, y + h], outline=edgecolor, width=linewidth)
    
    # Draw the bounding boxes with index at the least overlapping corner
    drawn_boxes = []
    for idx, bbox in enumerate(bboxes):
        text = str(idx)
        text_h, text_w = mark_helper.get_mark_size(text, image_height, image_width)
        corner_y, corner_x = _find_least_overlapping_corner(
            bbox, bboxes, drawn_boxes, (text_h, text_w), (image_height, image_width))
        
        # Define the index box (y, x, y + h, x + w)
        text_box = (corner_y, corner_x, text_h, text_w)

        if add_mark:
            # Draw the filled index box and text
            draw.rectangle([corner_x, corner_y, corner_x + text_w, corner_y + text_h], # (x, y, x + w, y + h)
                        fill="red")        
            font = mark_helper.get_font(image_height, image_width)
            draw.text((corner_x, corner_y), text, fill='white', font=font)
        
        # Update the list of drawn boxes
        drawn_boxes.append(np.array(text_box))
        
    if fn_save is not None: # PIL image
        image.save(fn_save)
    return image












In [5]:
def remove_overlap(boxes, iou_threshold):
    """Remove overlapping boxes, keeping the smaller one.
    Copied from utils.py to avoid OCR import at module level.
    Input/output: tensor of shape (N, 4) in xyxy normalized format.
    """
    def box_area(box):
        return (box[2] - box[0]) * (box[3] - box[1])

    def intersection_area(box1, box2):
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])
        return max(0, x2 - x1) * max(0, y2 - y1)

    def IoU(box1, box2):
        intersection = intersection_area(box1, box2)
        union = box_area(box1) + box_area(box2) - intersection + 1e-6
        if box_area(box1) > 0 and box_area(box2) > 0:
            ratio1 = intersection / box_area(box1)
            ratio2 = intersection / box_area(box2)
        else:
            ratio1, ratio2 = 0, 0
        return max(intersection / union, ratio1, ratio2)

    boxes = boxes.tolist()
    filtered = []
    for i, box1 in enumerate(boxes):
        is_valid = True
        for j, box2 in enumerate(boxes):
            if i != j and IoU(box1, box2) > iou_threshold and box_area(box1) > box_area(box2):
                is_valid = False
                break
        if is_valid:
            filtered.append(box1)
    return torch.tensor(filtered) if filtered else torch.zeros(0, 4)


def detect_ui_elements(image, yolo_model, box_threshold=None, overlap_iou_threshold=None, min_box_area=None):
    """Detect UI elements with YOLO (OmniParser).
    Applies overlap removal (keeps smaller box) and taskbar filter.
    Returns list of (y, x, h, w) normalized bboxes.

    Args:
        box_threshold: YOLO confidence threshold (default: module BOX_THRESHOLD=0.1)
        overlap_iou_threshold: IoU threshold for overlap removal (default: module OVERLAP_IOU_THRESHOLD=0.7)
        min_box_area: minimum normalized box area to keep (default: module MIN_BOX_AREA=0.0)
    """
    _box_threshold = box_threshold if box_threshold is not None else BOX_THRESHOLD
    _overlap_iou = overlap_iou_threshold if overlap_iou_threshold is not None else OVERLAP_IOU_THRESHOLD
    _min_area = min_box_area if min_box_area is not None else MIN_BOX_AREA

    w, h = image.size
    result = yolo_model.predict(source=image, conf=_box_threshold, iou=0.1, verbose=False)

    raw_boxes = result[0].boxes.xyxy
    if len(raw_boxes) == 0:
        return []

    # Normalize to [0, 1]
    xyxy_norm = raw_boxes / torch.Tensor([w, h, w, h]).to(raw_boxes.device)

    # Remove overlapping boxes (keep smaller box when IoU > threshold)
    xyxy_norm = remove_overlap(xyxy_norm, iou_threshold=_overlap_iou)

    # Convert xyxy -> (y, x, h, w), apply area filter and taskbar filter
    bboxes = []
    for box in xyxy_norm.tolist():
        x1, y1, x2, y2 = box
        bh, bw = y2 - y1, x2 - x1
        if bh * bw >= _min_area:
            center_y = (y1 + y2) / 2
            if center_y <= 0.93:  # Filter taskbar
                bboxes.append((y1, x1, bh, bw))
    return bboxes

In [ ]:


def get_xywh(input):
    x, y, w, h = input[0][0], input[0][1], input[2][0] - input[0][0], input[2][1] - input[0][1]
    x, y, w, h = int(x), int(y), int(w), int(h)
    return x, y, w, h

def get_xyxy(input):
    x, y, xp, yp = input[0][0], input[0][1], input[2][0], input[2][1]
    x, y, xp, yp = int(x), int(y), int(xp), int(yp)
    return x, y, xp, yp




def remove_overlap_new(boxes, iou_threshold, ocr_bbox=None):
    '''
    ocr_bbox format: [{'type': 'text', 'bbox':[x,y], 'interactivity':False, 'content':str }, ...]
    boxes format: [{'type': 'icon', 'bbox':[x,y], 'interactivity':True, 'content':None }, ...]

    '''
    assert ocr_bbox is None or isinstance(ocr_bbox, List)

    def box_area(box):
        return (box[2] - box[0]) * (box[3] - box[1])

    def intersection_area(box1, box2):
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])
        return max(0, x2 - x1) * max(0, y2 - y1)

    def IoU(box1, box2):
        intersection = intersection_area(box1, box2)
        union = box_area(box1) + box_area(box2) - intersection + 1e-6
        if box_area(box1) > 0 and box_area(box2) > 0:
            ratio1 = intersection / box_area(box1)
            ratio2 = intersection / box_area(box2)
        else:
            ratio1, ratio2 = 0, 0
        return max(intersection / union, ratio1, ratio2)

    def is_inside(box1, box2):
        # return box1[0] >= box2[0] and box1[1] >= box2[1] and box1[2] <= box2[2] and box1[3] <= box2[3]
        intersection = intersection_area(box1, box2)
        ratio1 = intersection / box_area(box1)
        return ratio1 > 0.80

    # boxes = boxes.tolist()
    filtered_boxes = []
    if ocr_bbox:
        filtered_boxes.extend(ocr_bbox)
    # print('ocr_bbox!!!', ocr_bbox)
    for i, box1_elem in enumerate(boxes):
        box1 = box1_elem['bbox']
        is_valid_box = True
        for j, box2_elem in enumerate(boxes):
            # keep the smaller box
            box2 = box2_elem['bbox']
            if i != j and IoU(box1, box2) > iou_threshold and box_area(box1) > box_area(box2):
                is_valid_box = False
                break
        if is_valid_box:
            if ocr_bbox:
                # keep yolo boxes + prioritize ocr label
                box_added = False
                ocr_labels = ''
                for box3_elem in ocr_bbox:
                    if not box_added:
                        box3 = box3_elem['bbox']
                        if is_inside(box3, box1): # ocr inside icon
                            # box_added = True
                            # delete the box3_elem from ocr_bbox
                            try:
                                # gather all ocr labels
                                ocr_labels += box3_elem['content'] + ' '
                                filtered_boxes.remove(box3_elem)
                            except:
                                continue
                            # break
                        elif is_inside(box1, box3): # icon inside ocr, don't added this icon box, no need to check other ocr bbox bc no overlap between ocr bbox, icon can only be in one ocr box
                            box_added = True
                            break
                        else:
                            continue
                if not box_added:
                    if ocr_labels:
                        filtered_boxes.append({'type': 'icon', 'bbox': box1_elem['bbox'], 'interactivity': True, 'content': ocr_labels, 'source':'box_yolo_content_ocr'})
                    else:
                        filtered_boxes.append({'type': 'icon', 'bbox': box1_elem['bbox'], 'interactivity': True, 'content': None, 'source':'box_yolo_content_yolo'})
            else:
                filtered_boxes.append(box1)
    return filtered_boxes # torch.tensor(filtered_boxes)





def check_ocr_box(image_source: str | Image.Image, display_img=True, output_bb_format='xywh', goal_filtering=None, easyocr_args=None, use_paddleocr=False):    
    if isinstance(image_source, str):
        image_source = Image.open(image_source)
    if image_source.mode == 'RGBA':
        # Convert RGBA to RGB to avoid alpha channel issues
        image_source = image_source.convert('RGB')
    image_np = np.array(image_source)
    w, h = image_source.size
    if use_paddleocr:
        if easyocr_args is None:
            text_threshold = 0.5
        else:
            text_threshold = easyocr_args['text_threshold']
        result = paddle_ocr.ocr(image_np)[0]
        coord = [item[0] for item in result if item[1][1] > text_threshold]
        text = [item[1][0] for item in result if item[1][1] > text_threshold]
    else:  # EasyOCR
        if easyocr_args is None:
            easyocr_args = {}
        result = reader.readtext(image_np, **easyocr_args)
        coord = [item[0] for item in result]
        text = [item[1] for item in result]
    if display_img:
        opencv_img = cv2.cvtColor(image_np, cv2.COLOR_RGB2BGR)
        bb = []
        for item in coord:
            x, y, a, b = get_xywh(item)
            bb.append((x, y, a, b))
            cv2.rectangle(opencv_img, (x, y), (x+a, y+b), (0, 255, 0), 2)
        #  matplotlib expects RGB
        plt.imshow(cv2.cvtColor(opencv_img, cv2.COLOR_BGR2RGB))
    else:
        if output_bb_format == 'xywh':
            bb = [get_xywh(item) for item in coord]
        elif output_bb_format == 'xyxy':
            bb = [get_xyxy(item) for item in coord]
    return (text, bb), goal_filtering

In [11]:
def annotate_single(image_path, yolo_model):
    image = Image.open(image_path).convert("RGB")
    width, height = image.size

    # =========================
    # 1. OCR (PIXEL xyxy)
    # =========================
    ocr_bbox_rslt, _ = check_ocr_box(
        image_path,
        display_img=False,
        output_bb_format='xyxy',
        easyocr_args={'paragraph': False, 'text_threshold': 0.9},
        use_paddleocr=False
    )

    ocr_text, ocr_bbox = ocr_bbox_rslt  # xyxy PIXEL

    ocr_bbox_elem = [
        {
            'type': 'text',
            'bbox': box,   # already pixel xyxy
            'interactivity': False,
            'content': txt
        }
        for box, txt in zip(ocr_bbox, ocr_text)
    ]

    # =========================
    # 2. FILTER LARGE OCR BOXES (IMPORTANT)
    # =========================
    def filter_large_ocr(boxes, max_area_ratio=0.05):
        filtered = []
        for b in boxes:
            x1, y1, x2, y2 = b['bbox']
            area = (x2 - x1) * (y2 - y1)
            if area < max_area_ratio * width * height:
                filtered.append(b)
        return filtered

    ocr_bbox_elem = filter_large_ocr(ocr_bbox_elem)

    # =========================
    # 3. YOLO → PIXEL xyxy
    # =========================
    bboxes = detect_ui_elements(
        image,
        yolo_model,
        box_threshold=BOX_THRESHOLD,
        overlap_iou_threshold=OVERLAP_IOU,
    )

    yolo_boxes_elem = []
    for (y, x, h, w) in bboxes:
        x1 = int(x * width)
        y1 = int(y * height)
        x2 = int((x + w) * width)
        y2 = int((y + h) * height)

        yolo_boxes_elem.append({
            'type': 'icon',
            'bbox': [x1, y1, x2, y2],
            'interactivity': True,
            'content': None
        })

    # =========================
    # 4. MERGE (OmniParser logic)
    # =========================
    filtered_boxes = remove_overlap_new(
        boxes=yolo_boxes_elem,
        iou_threshold=0.7,
        ocr_bbox=ocr_bbox_elem
    )

    if len(filtered_boxes) == 0:
        return image.copy(), {}, 0

    # =========================
    # 5. CONVERT → normalized (for your drawing)
    # =========================
    final_bboxes = []
    for box in filtered_boxes:
        x1, y1, x2, y2 = box['bbox']

        final_bboxes.append((
            y1 / height,
            x1 / width,
            (y2 - y1) / height,
            (x2 - x1) / width
        ))

    # optional limit
    if MAX_MARKS and len(final_bboxes) > MAX_MARKS:
        final_bboxes = final_bboxes[:MAX_MARKS]

    # =========================
    # 6. DRAW
    # =========================
    mark_helper = MarkHelper()
    mark_helper.min_font_size = MARK_FONT_MIN
    mark_helper.max_font_size = MARK_FONT_MAX

    annotated = plot_boxes_with_marks(
        image.copy(),
        final_bboxes,
        mark_helper,
        edgecolor=(255, 0, 0),
        linewidth=1,
        normalized_to_pixel=True,
        add_mark=True,
    )

    # =========================
    # 7. BUILD OUTPUT
    # =========================
    mark_to_center = {}

    for idx, (y, x, h, w) in enumerate(final_bboxes):
        x1 = int(x * width)
        y1 = int(y * height)
        x2 = int((x + w) * width)
        y2 = int((y + h) * height)

        mark_to_center[idx] = {
            "center": ((x1 + x2) // 2, (y1 + y2) // 2),
            "bbox": (x1, y1, x2, y2),
            "text": filtered_boxes[idx].get("content")  # OCR text if exists
        }

    return annotated, mark_to_center, len(final_bboxes)



os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Loading YOLO model: {OMNIPARSER_MODEL_PATH}")
yolo_model = YOLO(OMNIPARSER_MODEL_PATH)

# Scan testcase/ folder for test subfolders
test_folders = sorted([
    d for d in os.listdir(TESTCASE_DIR)
    if os.path.isdir(os.path.join(TESTCASE_DIR, d))
])
print(f"Found {len(test_folders)} test folders in {TESTCASE_DIR}")
print(f"Params: box_threshold={BOX_THRESHOLD}, overlap_iou={OVERLAP_IOU}, max_marks={MAX_MARKS}")

all_marks_info = {}
template = []
total_prompts = 0

for folder_name in test_folders:
    folder_path = os.path.join(TESTCASE_DIR, folder_name)

    # Find the image in this folder
    extensions = (".png", ".jpg", ".jpeg", ".bmp")
    images = [f for f in os.listdir(folder_path) if f.lower().endswith(extensions)]
    if not images:
        print(f"  WARNING: no image found in {folder_name}, skipping")
        continue
    img_name = images[0]
    img_path = os.path.join(folder_path, img_name)

    print(f"\n  [{folder_name}] {img_name}")

    # Annotate
    annotated, mark_to_center, num_marks = annotate_single(img_path, yolo_model)
    print(f"    Marks: {num_marks}")

    # Save annotated image (use folder name as filename for clarity)
    annotated_name = f"{folder_name}.png"
    annotated.save(os.path.join(OUTPUT_DIR, annotated_name))

    # Save marks info
    serializable = {}
    for mark_id, info in mark_to_center.items():
        serializable[str(mark_id)] = {
            "center_x": info["center"][0],
            "center_y": info["center"][1],
            "bbox": list(info["bbox"]),
        }
    all_marks_info[annotated_name] = {
        "num_marks": num_marks,
        "marks": serializable,
    }

    # Read prompts from the same folder
    prompt_files = sorted([
        f for f in os.listdir(folder_path)
        if f.startswith("prompt") and f.endswith(".txt")
    ])
    prompts_list = []
    for pf in prompt_files:
        with open(os.path.join(folder_path, pf)) as f:
            prompt_text = f.read().strip()
        if prompt_text:
            prompts_list.append({
                "prompt": prompt_text,
                "expected": {"ACTION": "CLICK", "MARK": 0, "VALUE": "None"},
            })

    total_prompts += len(prompts_list)
    print(f"    Prompts: {len(prompts_list)}")

    template.append({
        "test_folder": folder_name,
        "image": img_path,
        "annotated_image": os.path.join(OUTPUT_DIR, annotated_name),
        "num_marks": num_marks,
        "prompts": prompts_list,
    })

# Save marks info
marks_json_path = os.path.join(OUTPUT_DIR, "marks_info.json")
with open(marks_json_path, "w") as f:
    json.dump(all_marks_info, f, indent=2)

# Save test cases
with open(TEST_CASES_JSON, "w") as f:
    json.dump(template, f, indent=2)

print(f"\n{'='*60}")
print(f"Total: {len(test_folders)} images, {total_prompts} prompts")
print(f"Annotated images: {OUTPUT_DIR}")
print(f"Mark coordinates: {marks_json_path}")
print(f"Test cases:       {TEST_CASES_JSON}")
print(f"")
print(f"Next steps:")
print(f"  1. Open annotated images to see mark numbers")
print(f"  2. Edit {TEST_CASES_JSON} — fill in expected ACTION/MARK/VALUE")
print(f"  3. Run: python inference/run_e2e.py")
print(f"{'='*60}")

Loading YOLO model: /content/drive/MyDrive/Dataset/Testing/weights/icon_detect/model.pt
Found 2 test folders in /content/drive/MyDrive/Dataset/Testing/testcase
Params: box_threshold=0.25, overlap_iou=0.7, max_marks=100

  [test6] input.png
    Marks: 100
    Prompts: 5

  [test7] input.png
    Marks: 100
    Prompts: 5

Total: 2 images, 10 prompts
Annotated images: /content/drive/MyDrive/Dataset/Testing/annotated
Mark coordinates: /content/drive/MyDrive/Dataset/Testing/annotated/marks_info.json
Test cases:       /content/drive/MyDrive/Dataset/Testing/test_cases.json

Next steps:
  1. Open annotated images to see mark numbers
  2. Edit /content/drive/MyDrive/Dataset/Testing/test_cases.json — fill in expected ACTION/MARK/VALUE
  3. Run: python inference/run_e2e.py


In [24]:
import os
print(os.listdir("/content/drive/MyDrive/Dataset/Testing"))

['weights', 'arial.ttf', 'testcase']
